# 📓 Generative AI Module 1: Interactive Variational Autoencoder (VAE)
Welcome to Module 1 of our Generative Models course! In this notebook, we build a **Variational Autoencoder (VAE)** from scratch in PyTorch, train it on MNIST, and use interactive controls to explore its latent space and morph between digits.

---

## 💡 What is a VAE?
A standard Autoencoder (AE) compresses an input $x$ into a single bottleneck vector $z$. Because $z$ is unconstrained, standard autoencoders often leave gaps in the latent space, making them bad at generating new realistic data.

A **Variational Autoencoder (VAE)** fixes this by encoding $x$ as a **probability distribution** (a Gaussian described by mean $\mu$ and standard deviation $\sigma$) in the latent space:

$$z \sim \mathcal{N}(\mu, \sigma^2)$$

### The Reparameterization Trick
Since sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ is a stochastic operation, backpropagation cannot pass through it directly. We use the **reparameterization trick** to separate the deterministic parameters from random noise $\epsilon \sim \mathcal{N}(0, I)$:

$$z = \mu + \sigma \cdot \epsilon, \quad \text{where } \frac{\partial z}{\partial \mu} = 1, \quad \frac{\partial z}{\partial \sigma} = \epsilon$$

### The ELBO Loss Objective
The Evidence Lower BOund (ELBO) balances reconstruction quality and latent space regularization:

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{Reconstruction Loss (BCE)}} - \underbrace{\text{KL}(q_\phi(z|x) \parallel \mathcal{N}(0, I))}_{\text{Latent Regularization (KL Divergence)}}$$

![VAE Architecture](https://upload.wikimedia.org/wikipedia/commons/1/11/Reparameterization_trick_in_VAE.png)
*(Diagram: Reparameterization trick isolating noise $\epsilon$ to allow backpropagation through $\mu$ and $\sigma$.)*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Checkbox

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. VAE Model Definition ($z \in \mathbb{R}^2$): MLP vs. ConvVAE (CNN)

To compare performance and generation quality, we implement two selectable architectures using a boolean flag `use_conv`:

1. **Linear MLP (`use_conv=False`):** Flattened $28 \times 28$ image fed through fully-connected layers. Fast, but struggles with spatial structure and yields blurrier outputs.
2. **Convolutional VAE (`use_conv=True`):** Uses 2D Convolution (`Conv2d`) and Transposed Convolution (`ConvTranspose2d`) layers. Preserves 2D spatial locality and produces significantly **sharper images and cleaner digit reconstructions**!

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2, use_conv=False):
        super(VAE, self).__init__()
        self.use_conv = use_conv
        self.latent_dim = latent_dim
        
        if not use_conv:
            # --- 1. Linear MLP Architecture ---
            self.encoder_fc = nn.Sequential(
                nn.Linear(784, 400),
                nn.ReLU()
            )
            self.fc_mu = nn.Linear(400, latent_dim)
            self.fc_logvar = nn.Linear(400, latent_dim)
            
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, 400),
                nn.ReLU(),
                nn.Linear(400, 784),
                nn.Sigmoid()
            )
        else:
            # --- 2. Convolutional VAE (CNN) Architecture ---
            # Encoder: 28x28 -> 14x14 -> 7x7
            self.enc_conv = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1), # [B, 32, 14, 14]
                nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # [B, 64, 7, 7]
                nn.ReLU(),
                nn.Flatten(), # [B, 64 * 7 * 7 = 3136]
                nn.Linear(3136, 128),
                nn.ReLU()
            )
            self.fc_mu = nn.Linear(128, latent_dim)
            self.fc_logvar = nn.Linear(128, latent_dim)
            
            # Decoder: Latent -> 7x7 -> 14x14 -> 28x28
            self.dec_fc = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 3136),
                nn.ReLU()
            )
            self.dec_conv = nn.Sequential(
                nn.Unflatten(1, (64, 7, 7)),
                nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # [B, 32, 14, 14]
                nn.ReLU(),
                nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1), # [B, 1, 28, 28]
                nn.Sigmoid()
            )
            
    def encode(self, x):
        if not self.use_conv:
            x_flat = x.view(-1, 784)
            h = self.encoder_fc(x_flat)
        else:
            if x.dim() == 2: # handle flattened inputs
                x = x.view(-1, 1, 28, 28)
            h = self.enc_conv(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z):
        if not self.use_conv:
            return self.decoder(z)
        else:
            h = self.dec_fc(z)
            out = self.dec_conv(h)
            return out.view(-1, 784)
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

# Instantiate default model (change use_conv=True for CNN architecture)
use_conv_flag = True  # Toggle between False (Linear) and True (CNN)
model = VAE(latent_dim=2, use_conv=use_conv_flag).to(device)
print(f"Created VAE (use_conv={use_conv_flag}):\n{model}")

## 2. Loss Function (BCE + KL Divergence)
The loss is the sum of:
1. **Reconstruction Loss (Binary Cross-Entropy):** Measures pixel-level reconstruction accuracy.
2. **KL Divergence:** Measures deviation from standard normal prior $\mathcal{N}(0, I)$:

$$\text{D}_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, I)) = -\frac{1}{2} \sum \left( 1 + \log(\sigma^2) - \mu^2 - \sigma^2 \right)$$

In [ ]:
def loss_function(recon_x, x, mu, logvar):
    x_flat = x.view(-1, 784)
    BCE = F.binary_cross_entropy(recon_x, x_flat, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD, BCE, KLD

## 3. Training Function
We define a reusable function `train_vae` that allows students to train either the **Linear VAE** (`use_conv=False`) or the **Convolutional VAE** (`use_conv=True`) and compare their loss profiles.

In [ ]:
# Datasets & DataLoaders
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=128, shuffle=True)
test_loader = DataLoader(testset, batch_size=256, shuffle=False)

def train_vae(use_conv=False, epochs=5, lr=1e-3):
    vae_model = VAE(latent_dim=2, use_conv=use_conv).to(device)
    optimizer = torch.optim.Adam(vae_model.parameters(), lr=lr)
    model_name = "Convolutional VAE (CNN)" if use_conv else "Linear VAE (MLP)"
    print(f"=== Training {model_name} for {epochs} epochs ===")
    
    vae_model.train()
    for epoch in range(epochs):
        total_loss, total_bce, total_kld = 0, 0, 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            recon_batch, mu, logvar = vae_model(data)
            loss, bce, kld = loss_function(recon_batch, data, mu, logvar)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_bce += bce.item()
            total_kld += kld.item()
            
        avg_loss = total_loss / len(trainset)
        avg_bce = total_bce / len(trainset)
        avg_kld = total_kld / len(trainset)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.2f} (BCE: {avg_bce:.2f}, KLD: {avg_kld:.2f})")
        
    return vae_model

# Train the Convolutional VAE by default for high quality
conv_model = train_vae(use_conv=True, epochs=5)
# Optionally train the linear model to compare:
linear_model = train_vae(use_conv=False, epochs=5)

## 🎛️ Interactive Comparison Widget: MLP vs. ConvVAE Latent Navigation
Toggle the **`use_cnn`** checkbox to compare the visual sharpness of digits generated by the **Linear VAE** versus the **Convolutional VAE** at the exact same latent coordinates $(z_1, z_2)$!

In [ ]:
def explore_latent_space(z1=0.0, z2=0.0, use_cnn=True):
    selected_model = conv_model if use_cnn else linear_model
    selected_model.eval()
    model_title = "Convolutional VAE (CNN)" if use_cnn else "Linear VAE (MLP)"
    
    with torch.no_grad():
        z = torch.tensor([[z1, z2]], dtype=torch.float32).to(device)
        generated = selected_model.decode(z).cpu().view(28, 28)
        
        plt.figure(figsize=(4, 4))
        plt.imshow(generated, cmap='gray')
        plt.title(f"{model_title}\nz = ({z1:.2f}, {z2:.2f})")
        plt.axis('off')
        plt.show()

# Interactive Sliders with Model Toggle
interact(explore_latent_space,
         z1=FloatSlider(min=-3.0, max=3.0, step=0.1, value=0.0, description='z1 (Dim 1)'),
         z2=FloatSlider(min=-3.0, max=3.0, step=0.1, value=0.0, description='z2 (Dim 2)'),
         use_cnn=Checkbox(value=True, description='Use ConvVAE (CNN)'));

## 🖼️ Side-by-Side 2D Latent Manifold Canvas
Let's render a $20 \times 20$ grid across the latent space $[-3, 3]^2$ for both architectures so students can visually compare the image quality, sharpness, and digit transitions.

In [ ]:
def plot_side_by_side_manifold(conv_m, linear_m, n=15):
    conv_m.eval()
    linear_m.eval()
    grid_x = np.linspace(-3, 3, n)
    grid_y = np.linspace(-3, 3, n)
    
    fig_conv = np.zeros((28 * n, 28 * n))
    fig_linear = np.zeros((28 * n, 28 * n))
    
    with torch.no_grad():
        for i, yi in enumerate(grid_y):
            for j, xi in enumerate(grid_x):
                z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
                # Conv VAE
                decoded_c = conv_m.decode(z).cpu().view(28, 28).numpy()
                fig_conv[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = decoded_c
                # Linear VAE
                decoded_l = linear_m.decode(z).cpu().view(28, 28).numpy()
                fig_linear[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = decoded_l
                
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(fig_linear, cmap='gray')
    axes[0].set_title("Linear VAE (MLP) - Latent Manifold", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(fig_conv, cmap='gray')
    axes[1].set_title("Convolutional VAE (CNN) - Latent Manifold", fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

plot_side_by_side_manifold(conv_model, linear_model, n=15)

## 🎛️ Interactive Widget 2: Latent Morphing & Interpolation
Select two test set images $x_1$ and $x_2$. We encode them to get $\mu_1$ and $\mu_2$, then interpolate smoothly between them:

$$z_\alpha = (1 - \alpha) \mu_1 + \alpha \mu_2, \quad \alpha \in [0, 1]$$

In [ ]:
data_iter = iter(test_loader)
images, labels = next(data_iter)

img1, label1 = images[0], labels[0].item()
img2, label2 = images[1], labels[1].item()

def morph_digits(alpha=0.5, use_cnn=True):
    selected_model = conv_model if use_cnn else linear_model
    selected_model.eval()
    
    with torch.no_grad():
        mu1, _ = selected_model.encode(img1.unsqueeze(0).to(device))
        mu2, _ = selected_model.encode(img2.unsqueeze(0).to(device))
        
        z_alpha = (1 - alpha) * mu1 + alpha * mu2
        morphed_img = selected_model.decode(z_alpha).cpu().view(28, 28)
        
        fig, axes = plt.subplots(1, 3, figsize=(10, 3))
        axes[0].imshow(img1.squeeze(), cmap='gray')
        axes[0].set_title(f"Start (Digit {label1})")
        axes[0].axis('off')
        
        axes[1].imshow(morphed_img, cmap='gray')
        model_type = "CNN" if use_cnn else "MLP"
        axes[1].set_title(f"Morphed [{model_type}] (alpha = {alpha:.2f})")
        axes[1].axis('off')
        
        axes[2].imshow(img2.squeeze(), cmap='gray')
        axes[2].set_title(f"End (Digit {label2})")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

interact(morph_digits,
         alpha=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='Morph (alpha)'),
         use_cnn=Checkbox(value=True, description='Use ConvVAE (CNN)'));